In [6]:
import anndata as ad
import numpy as np
import os

import torch
import nicheformer
from transformers import AutoModelForMaskedLM, AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import snapshot_download 

In [7]:
from nicheformer.models import Nicheformer

repo_dir = snapshot_download(
    repo_id="theislab/Nicheformer",
    allow_patterns=[
        "config.json",
        "model.safetensors",
    ],
)

# here are the 
# * configuration_nicheformer.py
# * modeling_nicheformer.py
# files, install them to the model/ directory so that we can import them
if not os.path.exists("model"):
    os.makedirs("model")
os.system(f"cp {repo_dir}/configuration_nicheformer.py model/.")
os.system(f"cp {repo_dir}/modeling_nicheformer.py model/.")

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 43240.25it/s]


0

In [8]:
# get configuration and model from the downloaded files
from model.configuration_nicheformer import NicheformerConfig
from model.modeling_nicheformer import NicheformerForMaskedLM

# define a wrapper class to get embeddings from the model
class NicheformerTorchscript(torch.nn.Module):
    def __init__(self, model: torch.nn.Module):
        super().__init__()
        self.model = model

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        return self.model.get_embeddings(
            input_ids=input_ids,
            attention_mask=attention_mask,
            layer=-1,
            with_context=False,
        )

# configure the model and load the weights
config = NicheformerConfig.from_pretrained(repo_dir)
hf_model = NicheformerForMaskedLM(config)

state = load_file(f"{repo_dir}/model.safetensors")
missing, unexpected = hf_model.load_state_dict(state, strict=False)

print("missing:", missing)
print("unexpected:", unexpected)

hf_model.eval()
hf_model.requires_grad_(False)

wrapper = NicheformerTorchscript(hf_model).eval()

dummy_input_ids = torch.full((1, 1500), 3, dtype=torch.long)
dummy_attention_mask = torch.ones((1, 1500), dtype=torch.long)

with torch.inference_mode():
    eager_out = wrapper(dummy_input_ids, dummy_attention_mask)

    traced = torch.jit.trace(
        wrapper,
        (dummy_input_ids, dummy_attention_mask),
        strict=False,
    )
    traced.eval()

    traced_out = traced(dummy_input_ids, dummy_attention_mask)

torch.testing.assert_close(
    eager_out.detach().cpu(),
    traced_out.detach().cpu(),
    rtol=1e-4,
    atol=1e-4,
)

# Save UNFROZEN.
traced.save("nicheformer_embeddings.pt")

# Test Python reload.
reloaded = torch.jit.load("nicheformer_embeddings.pt", map_location="cpu")
reloaded.eval()

with torch.inference_mode():
    reload_out = reloaded(dummy_input_ids, dummy_attention_mask)

torch.testing.assert_close(
    eager_out.detach().cpu(),
    reload_out.detach().cpu(),
    rtol=1e-4,
    atol=1e-4,
)

print("saved: nicheformer_embeddings.pt")
print("n params:", sum(p.numel() for p in hf_model.parameters()))

missing: []
unexpected: []


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


saved: nicheformer_embeddings.pt
n params: 48955252
